# Step 7 - Explicit gender-bias detection: curated lexicon versus RoBERTa

**Task.** Detect explicit gender bias (taxonomy v0.3 codes E1 gender restriction or preference, E2 gender-marked occupational wording, E3 generic gendered pronoun) in the Philippine PESO-linked job advertisements, and compare (a) a curated lexicon with context rules and (b) a fine-tuned RoBERTa token tagger, against a context-free word-list baseline.

**Required outputs.** (1) a curated lexicon of explicit markers, (2) the comparison between the lexicon baseline and RoBERTa, (3) validation of precision and recall on a test subset, (4) precision, recall, F1 and false-positive rate.

**Conditions followed**
1. Methods and discussion are tied to literature from 2022-2025 (references in section 9). Where no verified 2022-2025 source exists, the text says so.
2. Test subset: a stratified random sample of 400 advertisements. The corpus holds 1,500 advertisements (not 1,501); all counts below use 1,500.
3. All parameters are printed in the cell that uses them and collected in `parameters.json`.
4. Results are interpreted against the literature (section 7) and limitations are stated with their likely effect on the results (section 8).

**Standing constraints.** The reference labels are the rule-derived (silver) labels of Steps 5 and 4b, not human annotations. Every score below measures agreement with those labels, not validity. No human annotation is used at this stage. Taxonomy v0.3 has not been formally adopted by the owner; its open decisions (Step 4b) are not applied here. Implicit codes (I1, I2) are out of scope and count as non-explicit.

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
import json, re, time, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 220)

OUT = Path("outputs") / "s7_explicit_detection"
OUT.mkdir(parents=True, exist_ok=True)

PARAMS = {
    "sample_n": 400, "sample_seed": 20261026, "strata": "ad route under taxonomy v0.3 (Step 5b)", "min_per_stratum": 2,
    "val_share_of_pool": 0.10, "val_seed": 42,
    "model": "roberta-base", "labels": ["O", "B-EXP", "I-EXP"], "seeds": [1, 2, 3],
    "epochs": 4, "batch": 32, "lr": 3e-5, "weight_decay": 0.01, "warmup_share": 0.10, "max_len": 96,
    "bootstrap_resamples": 1000, "bootstrap_seed": 7, "ci": "95% percentile, ads resampled with replacement",
    "match_rule_primary": "predicted span overlaps a silver explicit span by at least one character",
}
globals().update({k.upper(): v for k, v in PARAMS.items() if k in ("sample_n", "sample_seed", "seeds", "epochs", "batch", "lr", "max_len", "model", "labels")})
json.dump(PARAMS, open(OUT / "parameters.json", "w"), indent=1)
print(json.dumps(PARAMS, indent=1))

rd = lambda p: pd.read_csv(p, encoding="utf-8-sig")
units = rd("outputs/analysis/units.csv")
ad_ctx = rd("outputs/s5b_taxonomy_test/ad_context_v0.3.csv")
sp_all = rd("outputs/s5b_taxonomy_test/preannotation_spans_v0.3.csv")
ad_ctx["route_v03"] = ad_ctx["route_v03"].fillna("no span")
print(f"{units['record_id'].nunique():,} ads, {len(units):,} units, {len(sp_all):,} v0.3 spans")

{
 "sample_n": 400,
 "sample_seed": 20261026,
 "strata": "ad route under taxonomy v0.3 (Step 5b)",
 "min_per_stratum": 2,
 "val_share_of_pool": 0.1,
 "val_seed": 42,
 "model": "roberta-base",
 "labels": [
  "O",
  "B-EXP",
  "I-EXP"
 ],
 "seeds": [
  1,
  2,
  3
 ],
 "epochs": 4,
 "batch": 32,
 "lr": 3e-05,
 "weight_decay": 0.01,
 "warmup_share": 0.1,
 "max_len": 96,
 "bootstrap_resamples": 1000,
 "bootstrap_seed": 7,
 "ci": "95% percentile, ads resampled with replacement",
 "match_rule_primary": "predicted span overlaps a silver explicit span by at least one character"
}
1,500 ads, 7,971 units, 1,217 v0.3 spans


## 1. Test subset: stratified random sample of 400 advertisements

**Why stratify.** Explicit wording is rare in the corpus: 340 of 1,500 advertisements (22.7%) carry an explicit silver span. A simple random sample would leave the explicit stratum small, and precision and recall would rest on few positives. Stratifying by the advertisement's v0.3 route (explicit, implicit, none, unclear, no span) keeps every stratum represented at its corpus share. Allocation is proportional with a minimum of 2 per stratum, so the rarest stratum (unclear, 9 advertisements) is not empty.

**Why a held-out set at all.** Evaluation on advertisements that the tagger saw during training inflates scores; Kapoor and Narayanan [14] describe such leakage as a cause of non-reproducible results in machine-learning-based science. The 400 advertisements are therefore excluded from RoBERTa training and from lexicon curation.

**Parameters.** seed 20261026; strata = Step 5b ad route; proportional allocation, minimum 2 per stratum. No literature in 2022-2025 specific to stratified sampling of job advertisements was found, so the design rests on the reasoning above and on [14].

In [2]:
N = PARAMS["sample_n"]
sz = ad_ctx["route_v03"].value_counts()
raw = sz / len(ad_ctx) * N
alloc = np.floor(raw).astype(int).clip(lower=PARAMS["min_per_stratum"])
while alloc.sum() < N:
    alloc[(raw - alloc).idxmax()] += 1
while alloc.sum() > N:
    alloc[alloc.idxmax()] -= 1
rng = np.random.RandomState(PARAMS["sample_seed"])
test_ads = []
for k, g in ad_ctx.groupby("route_v03"):
    test_ads += list(rng.choice(sorted(g["record_id"]), alloc[k], replace=False))
test_ads = sorted(test_ads)
route_of = dict(zip(ad_ctx["record_id"], ad_ctx["route_v03"]))
employer_of = units.drop_duplicates("record_id").set_index("record_id")["employer_id"].to_dict()

tbl = pd.DataFrame({"corpus_ads": sz, "sampled_ads": alloc}).loc[["explicit", "implicit", "unclear", "none", "no span"]]
tbl["sampling_fraction"] = (tbl["sampled_ads"] / tbl["corpus_ads"]).round(3)
tbl.loc["total"] = [tbl["corpus_ads"].sum(), tbl["sampled_ads"].sum(), round(tbl["sampled_ads"].sum() / tbl["corpus_ads"].sum(), 3)]
display(tbl)
pd.DataFrame({"record_id": test_ads, "route_v03": [route_of[a] for a in test_ads]}).to_csv(OUT / "test_sample_400_ads.csv", index=False, encoding="utf-8-sig")

pool_ads = sorted(set(units["record_id"]) - set(test_ads))
test_units = units[units["record_id"].isin(test_ads)].reset_index(drop=True)
pool_units = units[units["record_id"].isin(pool_ads)].reset_index(drop=True)
print(f"Test: {len(test_ads)} ads, {len(test_units):,} units. Pool (training/validation/lexicon curation): {len(pool_ads)} ads, {len(pool_units):,} units.")

,corpus_ads,sampled_ads,sampling_fraction
route_v03,,,
explicit,340.0,91.0,0.268
implicit,344.0,92.0,0.267
unclear,9.0,2.0,0.222
none,72.0,19.0,0.264
no span,735.0,196.0,0.267
total,1500.0,400.0,0.267


Test: 400 ads, 2,142 units. Pool (training/validation/lexicon curation): 1100 ads, 5,829 units.


## 2. Reference labels and what they can show

The reference (gold) spans are the **explicit silver spans** of Step 5b: taxonomy v0.3 codes E1, E2 and E3 with valid offsets. G1 (non-restrictive listing), P, INC, U, I1 and I2 spans are not explicit and are negatives for this task. They come from lexicon rules (Steps 4-5), with the E1/E2/E3 split from the v0.3 crosswalk.

**Circularity, stated plainly.** The curated lexicon below is an independent re-implementation of the explicit rules, written from the codebook and the training pool, but it is designed by the same team on the same corpus as the silver labels. High agreement with silver labels is therefore expected and is not evidence that either is correct. Everitt et al. [23] evaluate bias detection against a gold set annotated by two people (76 hours for the real job descriptions) and compare prompting with fine-tuning; no such human gold set exists here, and Mihaljevic et al. [5] show that tools applied to the same advertisements can produce different assessments of exclusion. For these reasons all results in this notebook are reported as **agreement with silver labels**, and disagreements are listed for later review, not resolved.

In [3]:
gold_sp = sp_all[sp_all["expression_code"].isin(["E1", "E2", "E3"]) & sp_all["span_offset_valid"].astype(bool)].copy()
gold_by_unit = {}
for r in gold_sp.itertuples():
    gold_by_unit.setdefault(r.unit_id, []).append((int(r.span_start), int(r.span_end), r.expression_code))
for k in gold_by_unit:
    gold_by_unit[k].sort()
    assert all(b[0] >= a[1] for a, b in zip(gold_by_unit[k], gold_by_unit[k][1:])), "overlapping gold spans in " + k

other_by_unit = {}
for r in sp_all[~sp_all["expression_code"].isin(["E1", "E2", "E3"])].itertuples():
    if r.span_offset_valid:
        other_by_unit.setdefault(r.unit_id, []).append((int(r.span_start), int(r.span_end), r.expression_code))

tu_gold = test_units["unit_id"].map(lambda u: len(gold_by_unit.get(u, [])))
summ = pd.DataFrame([
    ("explicit silver spans (E1+E2+E3)", int(tu_gold.sum()), sum(len(v) for k, v in gold_by_unit.items() if k in set(pool_units["unit_id"]))),
    ("units with an explicit silver span", int((tu_gold > 0).sum()),     int(pool_units["unit_id"].isin(list(gold_by_unit)).sum())),
        ("ads with an explicit silver span", int(test_units[tu_gold > 0]["record_id"].nunique()), int(pool_units[pool_units["unit_id"].isin(list(gold_by_unit))]["record_id"].nunique())),
], columns=["measure", "test (400 ads)", "pool (1,100 ads)"])
display(summ)
cc = Counter(c for k, v in gold_by_unit.items() if k in set(test_units["unit_id"]) for _, _, c in v)
cp = Counter(c for k, v in gold_by_unit.items() if k in set(pool_units["unit_id"]) for _, _, c in v)
display(pd.DataFrame({"test": cc, "pool": cp}).fillna(0).astype(int).loc[["E1", "E2", "E3"]])

,measure,test (400 ads),"pool (1,100 ads)"
0,explicit silver spans (E1+E2+E3),99,263
1,units with an explicit silver span,96,259
2,ads with an explicit silver span,91,249


,test,pool
E1,86,231
E2,8,32
E3,5,0


## 3. Curated lexicon of explicit gender-bias markers

**Method.** Curation follows three steps, all on the **training pool only** (the 400 test advertisements were not read):
1. *Seed.* Gender terms named in the v0.3 codebook definitions (E1: restriction wording such as male, female; E2: gender-marked occupational wording such as salesgirl; E3: he/she pronouns), plus their plain inflections (men, women, lady, ladies, gentleman, boy, girl).
2. *Pool scan.* Every word in the pool that ends in a gendered morpheme (-man, -men, -woman, -lady, -girl, -boy) or equals a seed term is listed with its pool frequency, and each is kept or rejected with a stated reason.
3. *Context rules.* Single words cannot separate "Male" (a restriction) from "Male/Female" (a non-restrictive listing, G1), "he/she" (a pair) from "he" (generic pronoun), or the network acronym "MAN" from a gendered title. Rules R1-R5 below encode these distinctions from the v0.3 codebook.

**Why a lexicon with rules and not a bare word list.** Everitt et al. [23] note that regular expressions remain effective where bias-coded words are predictable (their example is age discrimination in Dutch job descriptions) but that find-and-replace does not scale well because individual words carry several meanings depending on context. Jiang et al. [8] built a Chinese gender lexicon for job advertisements on the premise that lexicons need adaptation to the language and market, which supports curating from our own corpus and not importing an English list unchanged. The explicit class is the case where this approach is most likely to work: the marking is lexical and the vocabulary is small.

**Context rules**
- **R1 pair or listing (G1).** If a unit names both a male-class and a female-class term, none is flagged ("Male/Female", "Male and Female", "Male, Female, LGBT").
- **R2 gender-conditioned requirement.** If the pair is tied to differing requirements ("5'4" for Male and 5'0" for Female"), the terms are flagged and the span includes the cue "for".
- **R3 acronym.** Upper-case "MAN" is not flagged when the unit also holds another network acronym (LAN, WAN, GAN, PAN).
- **R4 pronoun pair.** If a unit contains both a masculine and a feminine pronoun (he/she, his/her), none is flagged (G1). A lone pronoun is flagged E3.
- **R5 occupational suffix.** A word ending in -man, -men, -woman, -women, -lady, -ladies, -girl or -boy is flagged E2 unless it is on the stop list (not gendered in meaning).

In [4]:
TERM_CLASS = {  # E1 restriction terms; observed in pool (freq) or seed
    "male": "M", "males": "M", "men": "M", "gentleman": "M", "gentlemen": "M", "guy": "M", "guys": "M",
    "female": "F", "females": "F", "women": "F", "woman": "F", "lady": "F", "ladies": "F",
}
OCC_SUFFIX = re.compile(r"^[a-z]{2,}(?:man|men|woman|women|lady|ladies|girl|girls|boy|boys)$")
BARE_OCC = {"boy", "girl", "boys", "girls"}
STOP = {  # word -> reason
    "human": "not gendered (observed in pool)", "humans": "not gendered",
    "acumen": "not gendered (observed in pool)", "managemen": "typo of 'management' (observed in pool)",
    "german": "not gendered", "roman": "not gendered", "norman": "not gendered", "talisman": "not gendered",
    "ottoman": "not gendered", "shaman": "not gendered", "specimen": "not gendered", "regimen": "not gendered",
    "abdomen": "not gendered", "omen": "not gendered", "caveman": "not an occupation title", "cinema": "no match",
}
PRON_M, PRON_F = {"he", "him", "his", "himself"}, {"she", "her", "hers", "herself"}
NET_ACR = re.compile(r"\b(?:LAN|WAN|GAN|PAN)\b")
WORD = re.compile(r"[A-Za-z]+")


def lexicon_predict(text, component):
    hits = [(m.start(), m.end(), m.group().lower(), m.group()) for m in WORD.finditer(text)]
    out = []
    male = [h for h in hits if TERM_CLASS.get(h[2]) == "M"]
    fem = [h for h in hits if TERM_CLASS.get(h[2]) == "F"]
    cond = re.search(r"\bfor\s+(?:male|female|men|women)\b", text, re.I)
    if male and fem and not cond:
        pass
    else:
        for s, e, lw, raw in male + fem:
            if male and fem and cond:
                m = re.search(r"\bfor\s+" + re.escape(raw), text)
                s = m.start() if m else s
            out.append((s, e, "E1"))
    for s, e, lw, raw in hits:
        if lw in STOP or lw in TERM_CLASS:
            continue
        if lw == "man":
            if raw == "MAN" and NET_ACR.search(text):
                continue
            out.append((s, e, "E2"))
        elif lw in BARE_OCC or OCC_SUFFIX.match(lw):
            out.append((s, e, "E2"))
    pm = [h for h in hits if h[2] in PRON_M]
    pf = [h for h in hits if h[2] in PRON_F]
    if not (pm and pf):
        out += [(s, e, "E3") for s, e, lw, raw in pm + pf]
    return sorted(out)


NAIVE_WORDS = set(TERM_CLASS) | {"man", "boy", "girl", "boys", "girls"} | PRON_M | PRON_F
def naive_predict(text, component):
    out = []
    for m in WORD.finditer(text):
        lw = m.group().lower()
        if lw in NAIVE_WORDS or OCC_SUFFIX.match(lw):
            out.append((m.start(), m.end(), "X"))
    return out


# pool scan: every word that could be a lexicon entry, with frequency in the pool only
vocab = Counter(w.lower() for t in pool_units["text_masked"] for w in WORD.findall(str(t)))
cand = {w: n for w, n in vocab.items() if w in NAIVE_WORDS or w in STOP or OCC_SUFFIX.match(w) or re.search(r"(?:man|men)$", w)}
rows = []
for w, n in sorted(cand.items(), key=lambda x: -x[1]):
    if w in STOP:
        rows.append((w, n, "rejected", "-", STOP[w]))
    elif w in TERM_CLASS:
        rows.append((w, n, "kept", "E1", "gender term, R1/R2 apply"))
    elif w == "man":
        rows.append((w, n, "kept", "E2", "R3 (acronym) applies"))
    elif w in PRON_M | PRON_F:
        rows.append((w, n, "kept", "E3", "R4 applies"))
    elif w in BARE_OCC or OCC_SUFFIX.match(w):
        rows.append((w, n, "kept", "E2", "R5 occupational suffix"))
    else:
        rows.append((w, n, "rejected", "-", "ends in -man/-men but not an occupational suffix match"))
pool_scan = pd.DataFrame(rows, columns=["word", "pool_frequency", "decision", "class", "reason"])
display(pool_scan)
pool_scan.to_csv(OUT / "lexicon_pool_scan.csv", index=False, encoding="utf-8-sig")

,word,pool_frequency,decision,class,reason
0,male,228,kept,E1,"gender term, R1/R2 apply"
1,female,151,kept,E1,"gender term, R1/R2 apply"
2,human,23,rejected,-,not gendered (observed in pool)
3,foreman,18,kept,E2,R5 occupational suffix
4,salesman,11,kept,E2,R5 occupational suffix
5,warehouseman,4,kept,E2,R5 occupational suffix
6,leadman,3,kept,E2,R5 occupational suffix
7,tireman,2,kept,E2,R5 occupational suffix
8,man,2,kept,E2,R3 (acronym) applies
9,acumen,2,rejected,-,not gendered (observed in pool)


In [5]:
entries = []
for w, c in TERM_CLASS.items():
    entries.append((w, "E1", "male-class term" if c == "M" else "female-class term", "codebook seed / inflection", int(vocab.get(w, 0)), "R1, R2"))
for w in sorted(PRON_M | PRON_F):
    entries.append((w, "E3", "generic pronoun", "codebook seed (E3)", int(vocab.get(w, 0)), "R4"))
entries.append(("man", "E2", "bare word in a title or phrase", "pool-observed", int(vocab.get("man", 0)), "R3"))
for w in sorted(BARE_OCC):
    entries.append((w, "E2", "gender-marked occupational word", "pool-observed" if vocab.get(w, 0) else "codebook seed", int(vocab.get(w, 0)), "-"))
obs = sorted(w for w in vocab if OCC_SUFFIX.match(w) and w not in STOP and w not in TERM_CLASS and w not in BARE_OCC)
for w in obs:
    entries.append((w, "E2", "gender-marked occupational word", "pool-observed", int(vocab[w]), "R5"))
entries.append(("*man, *men, *woman, *women, *lady, *ladies, *girl, *boy (suffix pattern)", "E2", "pattern for unseen occupational titles", "codebook seed (salesgirl)", int(sum(vocab[w] for w in obs)), "R5, stop list"))
lexicon = pd.DataFrame(entries, columns=["entry", "class", "kind", "basis", "pool_frequency", "context_rules"])
display(lexicon)
lexicon.to_csv(OUT / "curated_explicit_lexicon.csv", index=False, encoding="utf-8-sig")
json.dump({"terms": TERM_CLASS, "pronouns_masculine": sorted(PRON_M), "pronouns_feminine": sorted(PRON_F),
           "occupational_suffix_pattern": OCC_SUFFIX.pattern, "bare_occupational": sorted(BARE_OCC),
           "stop_list": STOP, "rules": ["R1 pair/listing not flagged", "R2 gender-conditioned requirement flagged with cue 'for'",
                                         "R3 upper-case MAN with network acronym not flagged", "R4 pronoun pair not flagged",
                                         "R5 occupational suffix flagged unless stop-listed"],
           "curated_from": "training pool only (1,100 advertisements)", "version": "s7-v1"},
          open(OUT / "curated_explicit_lexicon.json", "w"), indent=1)
print(f"{len(lexicon)} lexicon rows; {len(obs)} occupational words observed in the pool: {obs}")

,entry,class,kind,basis,pool_frequency,context_rules
0,male,E1,male-class term,codebook seed / inflection,228,"R1, R2"
1,males,E1,male-class term,codebook seed / inflection,0,"R1, R2"
2,men,E1,male-class term,codebook seed / inflection,0,"R1, R2"
3,gentleman,E1,male-class term,codebook seed / inflection,0,"R1, R2"
4,gentlemen,E1,male-class term,codebook seed / inflection,0,"R1, R2"
5,guy,E1,male-class term,codebook seed / inflection,0,"R1, R2"
6,guys,E1,male-class term,codebook seed / inflection,0,"R1, R2"
7,female,E1,female-class term,codebook seed / inflection,151,"R1, R2"
8,females,E1,female-class term,codebook seed / inflection,0,"R1, R2"
9,women,E1,female-class term,codebook seed / inflection,0,"R1, R2"


39 lexicon rows; 12 occupational words observed in the pool: ['ductmen', 'foreman', 'instrumentman', 'leadman', 'loadingmen', 'portman', 'salesman', 'slingman', 'steelman', 'tireman', 'warehouseman', 'watchman']


## 4. Systems compared

| System | Description |
|---|---|
| **Naive word list** | Context-free baseline: every gender word, pronoun and -man/-woman/-lady/-girl/-boy word is flagged, with no stop list and no rules R1-R5. It shows what the context rules add. |
| **Curated lexicon** (section 3) | The lexicon with rules R1-R5. No training. |
| **RoBERTa** | `roberta-base` fine-tuned as a BIO token tagger with one entity type (EXPLICIT) over each unit (the unit is the input; the full advertisement is not used as context at this step). G1, P, INC, U, I1 and I2 spans are tagged O, so the model has to learn that "Male/Female" is not explicit. |

**Why a token tagger.** Span-level tagging of bias in job-hiring text is used by Raza et al. [26] (Nbias, token-level BIAS entities across domains that include job hiring); Everitt et al. [23] frame detection as classification and compare prompting with fine-tuning, and report that fine-tuning outperformed prompting for implicit bias. A tagger gives the span offsets needed to trace a flag to the text, the stated aim of the project.

**Why RoBERTa.** It is a standard pretrained encoder for token classification and is available offline here. Its original paper (Liu et al., 2019, arXiv:1907.11692) is outside the 2022-2025 window and is cited only to identify the model. The Step 6 DistilBERT tagger is not the comparator because it was trained on E1, E2, I1, I2 and INC under v0.2 codes.

**Training parameters.** model `roberta-base`; labels O / B-EXPLICIT / I-EXPLICIT; 3 seeds (1, 2, 3); 4 epochs; batch 32; learning rate 3e-5, linear decay with 10% warm-up; weight decay 0.01; maximum 96 tokens; best epoch chosen by validation exact-match micro-F1. Training pool: 1,100 advertisements outside the test set, 990 for training and 110 for validation, split by advertisement within strata. These settings are carried over from Step 6 except for the model and learning rate; they were not tuned.

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

LABELS = PARAMS["labels"]; L2I = {l: i for i, l in enumerate(LABELS)}
rng = np.random.RandomState(PARAMS["val_seed"])
pool_route = {a: route_of[a] for a in pool_ads}
val_ads = set()
for k in sorted(set(pool_route.values())):
    ids = sorted(a for a, r in pool_route.items() if r == k)
    n_val = max(1, int(round(len(ids) * PARAMS["val_share_of_pool"])))
    val_ads |= set(rng.choice(ids, n_val, replace=False))
train_ads = sorted(set(pool_ads) - val_ads)
tr_units = pool_units[pool_units["record_id"].isin(train_ads)].reset_index(drop=True)
va_units = pool_units[pool_units["record_id"].isin(val_ads)].reset_index(drop=True)
print(f"train {len(train_ads)} ads / {len(tr_units):,} units; val {len(val_ads)} ads / {len(va_units):,} units; test {len(test_ads)} ads / {len(test_units):,} units")
assert not (set(train_ads) | val_ads) & set(test_ads)

tok = AutoTokenizer.from_pretrained(PARAMS["model"])
MAXLEN = PARAMS["max_len"]

def encode(df):
    enc = tok(df["text_masked"].astype(str).tolist(), truncation=True, max_length=MAXLEN, return_offsets_mapping=True)
    feats, lost = [], 0
    for i, uid in enumerate(df["unit_id"]):
        g = gold_by_unit.get(uid, []); offs = enc["offset_mapping"][i]; lab = []
        for (s, e) in offs:
            if s == e:
                lab.append(-100); continue
            tag = "O"
            for a, b, k in g:
                if s < b and e > a:
                    tag = "B-EXP" if s <= a else "I-EXP"; break
            lab.append(L2I[tag])
        for a, b, k in g:
            if not any(o[1] > a and o[0] < b and o[0] != o[1] for o in offs):
                lost += 1
        feats.append({"input_ids": enc["input_ids"][i], "labels": lab, "offsets": offs})
    return feats, lost

F_tr, l1 = encode(tr_units); F_va, l2 = encode(va_units); F_te, l3 = encode(test_units)
lens = [len(f["input_ids"]) for f in F_tr + F_va + F_te]
print(f"tokens per unit: median {int(np.median(lens))}, max {max(lens)}; units at the {MAXLEN}-token limit: {sum(l >= MAXLEN for l in lens)}; gold spans lost to truncation: {l1 + l2 + l3}")

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), m), tok.pad_token_id); att = torch.zeros((len(batch), m), dtype=torch.long); lab = torch.full((len(batch), m), -100)
    for i, b in enumerate(batch):
        n_ = len(b["input_ids"]); ids[i, :n_] = torch.tensor(b["input_ids"]); att[i, :n_] = 1; lab[i, :n_] = torch.tensor(b["labels"])
    return ids, att, lab

def decode(offs, pred_ids):
    spans_, cur = [], None
    for (s, e), p in zip(offs, pred_ids):
        if s == e:
            continue
        tag = LABELS[p]
        if tag == "O":
            if cur: spans_.append(tuple(cur)); cur = None
            continue
        if tag == "B-EXP" or cur is None:
            if cur: spans_.append(tuple(cur))
            cur = [s, e]
        else:
            cur[1] = e
    if cur: spans_.append(tuple(cur))
    return spans_

@torch.no_grad()
def predict(model, feats):
    model.eval(); out = []
    for i in range(0, len(feats), 64):
        b = feats[i:i + 64]; ids, att, _ = collate(b)
        pr = model(input_ids=ids, attention_mask=att).logits.argmax(-1).tolist()
        for f, p in zip(b, pr):
            out.append(decode(f["offsets"], p[:len(f["offsets"])]))
    return out

def exact_micro_f1(pred, df):
    tp = fp = fn = 0
    for p, uid in zip(pred, df["unit_id"]):
        G = {(a, b) for a, b, _ in gold_by_unit.get(uid, [])}; P = set(p)
        tp += len(G & P); fp += len(P - G); fn += len(G - P)
    return 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0

D:\Dev\project-equal\equal-v2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train 990 ads / 5,253 units; val 110 ads / 576 units; test 400 ads / 2,142 units


tokens per unit: median 9, max 66; units at the 96-token limit: 0; gold spans lost to truncation: 0


In [7]:
def train_one(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = AutoModelForTokenClassification.from_pretrained(PARAMS["model"], num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I)
    opt = torch.optim.AdamW(model.parameters(), lr=PARAMS["lr"] if "lr" in PARAMS else LR, weight_decay=PARAMS["weight_decay"])
    steps = EPOCHS * ((len(F_tr) + BATCH - 1) // BATCH)
    warm = PARAMS["warmup_share"] * steps
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, (steps - s) / steps))
    best, best_state, log = -1, None, []
    for ep in range(EPOCHS):
        model.train(); order = np.random.permutation(len(F_tr)); tot = 0; t0 = time.time()
        for i in range(0, len(order), BATCH):
            ids, att, lab = collate([F_tr[j] for j in order[i:i + BATCH]])
            loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad(); tot += loss.item() * len(ids)
        vf = exact_micro_f1(predict(model, F_va), va_units)
        log.append({"seed": seed, "epoch": ep + 1, "train_loss": tot / len(F_tr), "val_exact_micro_f1": vf, "seconds": time.time() - t0})
        print(f"seed {seed} epoch {ep + 1}: loss {tot / len(F_tr):.4f}, val F1 {vf:.4f}, {time.time() - t0:.0f}s", flush=True)
        if vf > best:
            best = vf; best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, log

roberta_pred, logs = {}, []
for sd in PARAMS["seeds"]:
    f = OUT / f"roberta_test_predictions_seed{sd}.json"
    if f.exists():
        roberta_pred[sd] = {k: [tuple(x) for x in v] for k, v in json.load(open(f)).items()}
        print(f"seed {sd}: predictions loaded from {f.name} (training skipped)")
        continue
    m, lg = train_one(sd); logs += lg
    pr = predict(m, F_te)
    roberta_pred[sd] = {u: p for u, p in zip(test_units["unit_id"], pr)}
    json.dump(roberta_pred[sd], open(f, "w"))
    del m
if logs:
    pd.DataFrame(logs).to_csv(OUT / "roberta_training_log.csv", index=False)
    old = OUT / "roberta_training_log_all.csv"
    pd.DataFrame(logs).to_csv(old, mode="a", header=not old.exists(), index=False)
display(pd.read_csv(OUT / "roberta_training_log.csv").round(4) if (OUT / "roberta_training_log.csv").exists() else "training log not available (predictions were cached)")

seed 1: predictions loaded from roberta_test_predictions_seed1.json (training skipped)
seed 2: predictions loaded from roberta_test_predictions_seed2.json (training skipped)
seed 3: predictions loaded from roberta_test_predictions_seed3.json (training skipped)


,seed,epoch,train_loss,val_exact_micro_f1,seconds
0,1,1,0.1630,0.7600,597.6940
1,1,2,0.0051,0.8966,618.2980
2,1,3,0.0020,0.9474,573.8647
3,1,4,0.0012,0.9455,585.0834
4,2,1,0.0947,0.7308,551.6104
5,2,2,0.0045,0.9286,568.9580
6,2,3,0.0013,0.8929,532.2931
7,2,4,0.0012,0.9286,499.9710
8,3,1,0.0921,0.7308,496.2265
9,3,2,0.0042,0.7547,495.9487


## 5. Evaluation design

**Levels.** (a) *Advertisement:* positive if it holds an explicit silver span; predicted positive if any span is predicted. (b) *Unit:* the same for each sentence/requirement unit. (c) *Span, overlap match:* a predicted span matches a silver span when they share at least one character (primary, because silver spans such as "for Male" can include a cue word that a tagger may not). (d) *Span, exact match:* identical offsets. (e) *Token:* each token (`\w+|[^\w\s]`) is positive if it lies in an explicit span.

**Metrics.** Precision = TP/(TP+FP); recall = TP/(TP+FN); F1 = 2PR/(P+R); false-positive rate (FPR) = FP/(FP+TN). FPR is defined where negatives exist (advertisement, unit, token). At span level there is no count of negative spans, so FPR is reported at the unit and token levels instead.

**Uncertainty.** 95% percentile intervals from 1,000 bootstrap resamples of the 400 advertisements (seed 7). RoBERTa values are the mean over 3 seeds; the seed range is shown separately. Differences between systems are bootstrapped on the same resamples (paired).

**Employer overlap.** The test sample is random over advertisements, so employers can appear in both the training pool and the test set; templates repeated by one employer can inflate RoBERTa scores [14]. Results are therefore also reported separately for test advertisements whose employer is, and is not, in the training pool.

In [8]:
TOK = re.compile(r"\w+|[^\w\s]")
LEVELS = ["ad", "unit", "span_overlap", "span_exact", "token"]
ad_index = {a: i for i, a in enumerate(test_ads)}

def per_ad_counts(pred):
    A = np.zeros((len(test_ads), len(LEVELS), 4))
    ad_gold = np.zeros(len(test_ads), bool); ad_pred = np.zeros(len(test_ads), bool)
    for r in test_units.itertuples():
        i = ad_index[r.record_id]
        g = gold_by_unit.get(r.unit_id, []); p = pred.get(r.unit_id, [])
        p = [(x[0], x[1]) for x in p]
        gp, pp = len(g) > 0, len(p) > 0
        ad_gold[i] |= gp; ad_pred[i] |= pp
        A[i, 1] += [gp and pp, (not gp) and pp, gp and (not pp), (not gp) and (not pp)]
        tp = sum(any(a < e and s < b for s, e in p) for a, b, _ in g)
        fp = sum(not any(a < e and s < b for a, b, _ in g) for s, e in p)
        A[i, 2] += [tp, fp, len(g) - tp, 0]
        G, P = {(a, b) for a, b, _ in g}, set(p)
        A[i, 3] += [len(G & P), len(P - G), len(G - P), 0]
        for m in TOK.finditer(str(r.text_masked)):
            tg = any(m.start() < b and m.end() > a for a, b, _ in g)
            tp_ = any(m.start() < e and m.end() > s for s, e in p)
            A[i, 4] += [tg and tp_, (not tg) and tp_, tg and (not tp_), (not tg) and (not tp_)]
    A[:, 0] = np.stack([ad_gold & ad_pred, (~ad_gold) & ad_pred, ad_gold & (~ad_pred), (~ad_gold) & (~ad_pred)], 1)
    return A

def prfr(c):
    tp, fp, fn, tn = c
    p = tp / (tp + fp) if tp + fp else np.nan
    r = tp / (tp + fn) if tp + fn else np.nan
    f = 2 * p * r / (p + r) if p == p and r == r and p + r else np.nan
    fpr = fp / (fp + tn) if fp + tn else np.nan
    return np.array([p, r, f, fpr])

lex_pred = {r.unit_id: lexicon_predict(str(r.text_masked), r.component) for r in test_units.itertuples()}
naive_pred = {r.unit_id: naive_predict(str(r.text_masked), r.component) for r in test_units.itertuples()}
SYS = {"Naive word list": [per_ad_counts(naive_pred)], "Curated lexicon": [per_ad_counts(lex_pred)],
       "RoBERTa (mean of 3 seeds)": [per_ad_counts(roberta_pred[s]) for s in PARAMS["seeds"]]}
SPRED = {"Naive word list": [naive_pred], "Curated lexicon": [lex_pred], "RoBERTa (mean of 3 seeds)": [roberta_pred[s] for s in PARAMS["seeds"]]}

def metric_tab(Alist, idx=None):
    vals = []
    for A in Alist:
        S = A.sum(0) if idx is None else A[idx].sum(0)
        vals.append(np.stack([prfr(S[l]) for l in range(len(LEVELS))]))
    return np.mean(vals, 0), np.array(vals)

rngb = np.random.RandomState(PARAMS["bootstrap_seed"])
B = PARAMS["bootstrap_resamples"]; boots = rngb.randint(0, len(test_ads), (B, len(test_ads)))
BOOT = {k: np.stack([metric_tab(v, ix)[0] for ix in boots]) for k, v in SYS.items()}
rows = []
for k, v in SYS.items():
    pt, seeds_ = metric_tab(v)
    for li, lv in enumerate(LEVELS):
        for mi, mn in enumerate(["precision", "recall", "f1", "fpr"]):
            if lv.startswith("span") and mn == "fpr":
                continue
            lo, hi = np.nanpercentile(BOOT[k][:, li, mi], [2.5, 97.5])
            rows.append({"system": k, "level": lv, "metric": mn, "value": pt[li, mi], "ci_low": lo, "ci_high": hi,
                         "seed_min": np.nanmin(seeds_[:, li, mi]) if len(v) > 1 else np.nan, "seed_max": np.nanmax(seeds_[:, li, mi]) if len(v) > 1 else np.nan})
res = pd.DataFrame(rows)
res.to_csv(OUT / "results_long.csv", index=False, encoding="utf-8-sig")
wide = res.assign(txt=lambda d: d.apply(lambda r: f"{r.value:.3f} ({r.ci_low:.3f}-{r.ci_high:.3f})", axis=1)).pivot_table(index=["level", "system"], columns="metric", values="txt", aggfunc="first")[["precision", "recall", "f1", "fpr"]]
wide = wide.reindex(LEVELS, level=0)
display(wide)
wide.to_csv(OUT / "results_table.csv", encoding="utf-8-sig")

cnt = pd.DataFrame([{"system": k, "level": lv, "TP": np.mean([A.sum(0)[li, 0] for A in v]), "FP": np.mean([A.sum(0)[li, 1] for A in v]),
                     "FN": np.mean([A.sum(0)[li, 2] for A in v]), "TN": np.mean([A.sum(0)[li, 3] for A in v])}
                    for k, v in SYS.items() for li, lv in enumerate(LEVELS)])
display(cnt.round(1))
cnt.to_csv(OUT / "confusion_counts.csv", index=False, encoding="utf-8-sig")

metric                                            precision               recall                   f1                  fpr
level        system                                                                                                       
ad           Curated lexicon            0.883 (0.820-0.940)  1.000 (1.000-1.000)  0.938 (0.901-0.969)  0.039 (0.019-0.060)
             Naive word list            0.689 (0.607-0.767)  1.000 (1.000-1.000)  0.816 (0.755-0.868)  0.133 (0.096-0.172)
             RoBERTa (mean of 3 seeds)  0.939 (0.889-0.982)  0.960 (0.916-0.991)  0.949 (0.913-0.975)  0.018 (0.005-0.034)
unit         Curated lexicon            0.879 (0.815-0.933)  0.979 (0.933-1.000)  0.926 (0.884-0.958)  0.006 (0.003-0.010)
             Naive word list            0.653 (0.558-0.739)  1.000 (1.000-1.000)  0.790 (0.716-0.850)  0.025 (0.018-0.034)
             RoBERTa (mean of 3 seeds)  0.928 (0.873-0.973)  0.931 (0.859-0.983)  0.929 (0.883-0.964)  0.003 (0.001-0.006)
span_overlap Curated lexicon            0.880 (0.816-0.935)  0.960 (0.878-1.000)  0.918 (0.868-0.957)                  NaN
             Naive word list            0.553 (0.453-0.654)  1.000 (1.000-1.000)  0.712 (0.624-0.791)                  NaN
             RoBERTa (mean of 3 seeds)  0.919 (0.863-0.963)  0.906 (0.795-0.983)  0.912 (0.840-0.963)                  NaN
span_exact   Curated lexicon            0.824 (0.748-0.890)  0.899 (0.807-0.968)  0.860 (0.796-0.913)                  NaN
             Naive word list            0.508 (0.417-0.602)  0.919 (0.860-0.968)  0.655 (0.569-0.732)                  NaN
             RoBERTa (mean of 3 seeds)  0.903 (0.844-0.951)  0.892 (0.767-0.977)  0.897 (0.813-0.955)                  NaN
token        Curated lexicon            0.882 (0.818-0.938)  0.874 (0.776-0.964)  0.878 (0.820-0.926)  0.001 (0.000-0.001)
             Naive word list            0.553 (0.453-0.654)  0.892 (0.800-0.965)  0.683 (0.592-0.759)  0.005 (0.004-0.007)
             RoBERTa (mean of 3 seeds)  0.894 (0.823-0.962)  0.868 (0.747-0.974)  0.878 (0.778-0.957)  0.001 (0.000-0.002)

,system,level,TP,FP,FN,TN
0,Naive word list,ad,91.0,41.0,0.0,268.0
1,Naive word list,unit,96.0,51.0,0.0,1995.0
2,Naive word list,span_overlap,99.0,80.0,0.0,0.0
3,Naive word list,span_exact,91.0,88.0,8.0,0.0
4,Naive word list,token,99.0,80.0,12.0,15026.0
5,Curated lexicon,ad,91.0,12.0,0.0,297.0
6,Curated lexicon,unit,94.0,13.0,2.0,2033.0
7,Curated lexicon,span_overlap,95.0,13.0,4.0,0.0
8,Curated lexicon,span_exact,89.0,19.0,10.0,0.0
9,Curated lexicon,token,97.0,13.0,14.0,15093.0


### 5a. Paired comparison: RoBERTa minus curated lexicon

Difference in F1 (and in FPR where defined) on the same 1,000 resamples. An interval that includes 0 means the data do not separate the two systems.

In [9]:
rows = []
for li, lv in enumerate(LEVELS):
    for mi, mn in enumerate(["f1", "fpr"]):
        if lv.startswith("span") and mn == "fpr":
            continue
        d = BOOT["RoBERTa (mean of 3 seeds)"][:, li, 2 + mi] - BOOT["Curated lexicon"][:, li, 2 + mi]
        pt = metric_tab(SYS["RoBERTa (mean of 3 seeds)"])[0][li, 2 + mi] - metric_tab(SYS["Curated lexicon"])[0][li, 2 + mi]
        lo, hi = np.nanpercentile(d, [2.5, 97.5])
        rows.append({"level": lv, "metric": mn, "roberta_minus_lexicon": pt, "ci_low": lo, "ci_high": hi, "interval_excludes_0": not (lo <= 0 <= hi)})
paired = pd.DataFrame(rows)
display(paired.round(4))
paired.to_csv(OUT / "paired_difference_roberta_vs_lexicon.csv", index=False, encoding="utf-8-sig")

,level,metric,roberta_minus_lexicon,ci_low,ci_high,interval_excludes_0
0,ad,f1,0.0111,-0.0231,0.0451,False
1,ad,fpr,-0.0205,-0.0383,-0.0053,True
2,unit,f1,0.0028,-0.0305,0.0364,False
3,unit,fpr,-0.0029,-0.0056,-0.0006,True
4,span_overlap,f1,-0.0059,-0.0481,0.0335,False
5,span_exact,f1,0.0373,-0.0265,0.1010,False
6,token,f1,-0.0001,-0.0784,0.0644,False
7,token,fpr,-0.0000,-0.0007,0.0009,False


## 6. Where the systems agree and disagree with the silver labels

### 6a. Recall by silver code, and by whether the wording occurs in the training pool

In [10]:
pool_gold_words = Counter()
pu = set(pool_units["unit_id"]); txt_of = dict(zip(units["unit_id"], units["text_masked"].astype(str)))
for uid, g in gold_by_unit.items():
    if uid in pu:
        for a, b, c in g:
            pool_gold_words[txt_of[uid][a:b].lower()] += 1

def gold_recall(pred_list):
    rows = []
    for uid in test_units["unit_id"]:
        for a, b, c in gold_by_unit.get(uid, []):
            w = txt_of[uid][a:b].lower()
            hit = np.mean([any(s < b and a < e for s, e, *_ in p.get(uid, [])) for p in pred_list])
            rows.append({"unit_id": uid, "code": c, "wording": w, "seen_in_pool_gold": w in pool_gold_words, "hit": hit})
    return pd.DataFrame(rows)

rec_rows = []
for k, pl in SPRED.items():
    d = gold_recall(pl); d["system"] = k
    rec_rows.append(d)
rec_all = pd.concat(rec_rows)
by_code = rec_all.groupby(["system", "code"]).agg(gold_spans=("hit", "size"), recall=("hit", "mean")).reset_index().pivot(index="code", columns="system", values="recall")
by_code.insert(0, "gold_spans", rec_all[rec_all.system == "Curated lexicon"].groupby("code").size())
display(by_code.round(3))
by_seen = rec_all.groupby(["system", "seen_in_pool_gold"]).agg(gold_spans=("hit", "size"), recall=("hit", "mean")).reset_index()
display(by_seen.round(3))
by_code.to_csv(OUT / "recall_by_silver_code.csv", encoding="utf-8-sig")
by_seen.to_csv(OUT / "recall_by_seen_wording.csv", index=False, encoding="utf-8-sig")

system,gold_spans,Curated lexicon,Naive word list,RoBERTa (mean of 3 seeds)
code,,,,
E1,86,1.0,1.0,0.977
E2,8,1.0,1.0,0.708
E3,5,0.2,1.0,0.000


,system,seen_in_pool_gold,gold_spans,recall
0,Curated lexicon,False,7,0.429
1,Curated lexicon,True,92,1.000
2,Naive word list,False,7,1.000
3,Naive word list,True,92,1.000
4,RoBERTa (mean of 3 seeds),False,7,0.048
5,RoBERTa (mean of 3 seeds),True,92,0.971


### 6b. Employer overlap: test advertisements whose employer is, and is not, in the training pool

In [11]:
train_emp = {employer_of[a] for a in pool_ads}
seen_mask = np.array([employer_of[a] in train_emp for a in test_ads])
rows = []
for name, mk in (("employer in training pool", seen_mask), ("employer not in training pool", ~seen_mask)):
    ix = np.where(mk)[0]
    for k, v in SYS.items():
        pt, _ = metric_tab(v, ix)
        for li, lv in enumerate(LEVELS):
            if lv in ("ad", "unit", "span_overlap"):
                rows.append({"subset": name, "ads": len(ix), "ads_with_explicit_silver": int(sum(A[ix, 0, 0].sum() + A[ix, 0, 2].sum() for A in v[:1])),
                             "system": k, "level": lv, "precision": pt[li, 0], "recall": pt[li, 1],                              "f1": pt[li, 2], "fpr": np.nan if lv.startswith("span") else pt[li, 3]})
emp = pd.DataFrame(rows)
display(emp[emp.level.isin(["ad", "span_overlap"])].round(3))
emp.to_csv(OUT / "employer_overlap_results.csv", index=False, encoding="utf-8-sig")

,subset,ads,ads_with_explicit_silver,system,level,precision,recall,f1,fpr
0,employer in training pool,362,77,Naive word list,ad,0.688,1.000,0.815,0.123
2,employer in training pool,362,77,Naive word list,span_overlap,0.541,1.000,0.702,NaN
3,employer in training pool,362,77,Curated lexicon,ad,0.906,1.000,0.951,0.028
5,employer in training pool,362,77,Curated lexicon,span_overlap,0.899,1.000,0.947,NaN
6,employer in training pool,362,77,RoBERTa (mean of 3 seeds),ad,0.978,0.970,0.974,0.006
8,employer in training pool,362,77,RoBERTa (mean of 3 seeds),span_overlap,0.967,0.958,0.962,NaN
9,employer not in training pool,38,14,Naive word list,ad,0.700,1.000,0.824,0.250
11,employer not in training pool,38,14,Naive word list,span_overlap,0.613,1.000,0.760,NaN
12,employer not in training pool,38,14,Curated lexicon,ad,0.778,1.000,0.875,0.167
14,employer not in training pool,38,14,Curated lexicon,span_overlap,0.789,0.789,0.789,NaN


### 6c. False positives: what the flagged-but-not-silver spans are

A predicted span with no silver explicit span is a false positive in the tables above. Here each one is checked against the silver labels that **are** present in the same place: G1 (non-restrictive listing), U (unclear), P, I1/I2, or no silver span at all. A false positive on a G1 or U span is a disagreement about the label, not necessarily a detector error; a false positive with no silver span is a candidate the Step 5 rules never flagged. None of these is resolved here.

In [12]:
def fp_breakdown(pred, seed_label):
    rows = []
    for r in test_units.itertuples():
        g = gold_by_unit.get(r.unit_id, []); oth = other_by_unit.get(r.unit_id, [])
        for p in pred.get(r.unit_id, []):
            s, e = p[0], p[1]
            if any(a < e and s < b for a, b, _ in g):
                continue
            hit = [c for a, b, c in oth if a < e and s < b]
            rows.append({"system": seed_label, "unit_id": r.unit_id, "record_id": r.record_id, "component": r.component,
                         "span": str(r.text_masked)[s:e], "silver_label_at_span": "+".join(sorted(set(hit))) if hit else "no silver span", "text": str(r.text_masked)[:140]})
    return pd.DataFrame(rows)

fpb = pd.concat([fp_breakdown(naive_pred, "Naive word list"), fp_breakdown(lex_pred, "Curated lexicon"),
                 fp_breakdown(roberta_pred[PARAMS["seeds"][0]], f"RoBERTa seed {PARAMS['seeds'][0]}")], ignore_index=True)
fpb.to_csv(OUT / "false_positive_spans.csv", index=False, encoding="utf-8-sig")
display(fpb.groupby(["system", "silver_label_at_span"]).size().rename("spans").reset_index().pivot(index="silver_label_at_span", columns="system", values="spans").fillna(0).astype(int))
print("Curated lexicon: predicted spans with no silver span, by wording")
display(fpb[(fpb.system == "Curated lexicon") & (fpb.silver_label_at_span == "no silver span")].groupby("span").size().rename("n").reset_index().sort_values("n", ascending=False))
print("Curated lexicon: predicted spans on a silver U or G1 span")
display(fpb[(fpb.system == "Curated lexicon") & (fpb.silver_label_at_span != "no silver span")][["unit_id", "span", "silver_label_at_span", "text"]].head(25))

system,Curated lexicon,Naive word list,RoBERTa seed 1
silver_label_at_span,,,
G1,0,58,0
U,6,7,4
no silver span,7,15,8


Curated lexicon: predicted spans with no silver span, by wording


,span,n
1,Steelman,2
2,Stockman,2
0,Lubeman,1
3,Tireman,1
4,Warehouseman,1


Curated lexicon: predicted spans on a silver U or G1 span


,unit_id,span,silver_label_at_span,text
80,row-0012-s002,Male,U,Male
81,row-0013-s002,Male,U,Male
82,row-0014-s002,Female,U,Female
83,row-0015-s002,Female,U,Female
90,row-0652-s003,female,U,"With pleasing personality, female and single"
91,row-1076-s003,female,U,"With pleasing personality, female and single"


### 6d. False negatives: silver explicit spans each system misses

In [13]:
rows = []
for sname, pl in (("Curated lexicon", [lex_pred]), ("RoBERTa seed 1", [roberta_pred[1]]), ("RoBERTa seed 2", [roberta_pred[2]]), ("RoBERTa seed 3", [roberta_pred[3]])):
    p = pl[0]
    for r in test_units.itertuples():
        for a, b, c in gold_by_unit.get(r.unit_id, []):
            if not any(s < b and a < e for s, e, *_ in p.get(r.unit_id, [])):
                rows.append({"system": sname, "unit_id": r.unit_id, "code": c, "silver_span": str(r.text_masked)[a:b], "text": str(r.text_masked)[:140]})
fn_df = pd.DataFrame(rows)
fn_df.to_csv(OUT / "false_negative_spans.csv", index=False, encoding="utf-8-sig")
display(fn_df.groupby(["system", "code", "silver_span"]).size().rename("n").reset_index())

,system,code,silver_span,n
0,Curated lexicon,E3,her,2
1,Curated lexicon,E3,his,2
2,RoBERTa seed 1,E2,MAN,1
3,RoBERTa seed 1,E2,Saleslady,1
4,RoBERTa seed 1,E3,he,1
5,RoBERTa seed 1,E3,her,2
6,RoBERTa seed 1,E3,his,2
7,RoBERTa seed 2,E1,Female SRs must not be pregnant,1
8,RoBERTa seed 2,E1,for Female,1
9,RoBERTa seed 2,E1,for Male,1


## 7. Interpretation: comparison with the literature

## 8. Interpreting the results and comparing with other papers

**Scope of every number below.** All scores measure agreement with the rule-derived (silver) Step 5 / taxonomy v0.3 labels on 400 ads. They are not accuracy against human judgment. No human gold exists in this stage by design.

### 8.1 What we found

1. **Context rules, not the word list, drive the lexicon's precision.** The naive word list reached ad-level precision 0.689 (95% CI 0.607-0.767) and span-level precision 0.553. 58 of its 80 false-positive spans sit on a silver G1 span (nonrestrictive "Male/Female" listings). The curated lexicon suppresses G1 by rule R1 and rose to ad-level precision 0.883 (0.820-0.940) and span-level 0.880 (0.816-0.935), with 0 false positives on G1. This agrees with the qualitative argument in Everitt et al. (2024) that plain find-and-replace is unreliable because words are polysemous, and with Jiang et al. (2023) that lexicons need adaptation to the local corpus. We did not compare numbers with those papers: their tasks and labels differ.
2. **RoBERTa and the curated lexicon are statistically indistinguishable on F1.** Ad level: 0.949 (0.913-0.975) vs 0.938 (0.901-0.969). Span (overlap): 0.912 (0.840-0.963) vs 0.918 (0.868-0.957). The paired RoBERTa-minus-lexicon F1 intervals include 0 at every level (section 5).
3. **RoBERTa has a lower false-positive rate.** The paired difference is -0.021 at ad level (CI -0.038 to -0.005) and -0.003 at unit level, both excluding 0. It buys this with lower recall (ad 0.960 vs 1.000), so the F1 gain is not significant.
4. **The two systems fail on different things.** Per silver code, recall is E1 1.000 (lexicon) vs 0.977 (RoBERTa), E2 1.000 vs 0.708, and E3 0.200 vs 0.000. The lexicon misses the 4 "his/her" pair spans because rule R4 suppresses pronoun pairs while the silver labels keep them (the open taxonomy decision on pairs). RoBERTa cannot learn E3: all 5 E3 spans are in the test set and none are in training. On wording not seen in the pool, recall is 0.429 for the lexicon and 0.048 for RoBERTa (7 spans), so the lexicon generalises better to new occupational words.
5. **Most lexicon "false positives" are disagreements with the silver labels, not clear errors.** Of its 13 false-positive spans, 7 are "-man" job titles that the Step 5 rules never flagged (Steelman 2, Stockman 2, Lubeman, Tireman, Warehouseman) and 6 fall on silver U spans (a "Male"/"Female" label in a non-requirement block and "female and single"). Earlier checks (s4b section 7) already showed 22 of 51 gender-marked "-man" titles without an E2 span. Under a revised silver standard, lexicon precision would likely rise. We do not claim this: it needs human review.

Our results suggest that a context-aware lexicon is not associated with poorer detection than a fine-tuned RoBERTa on this corpus, when the labels are rule-derived. The proposed lexicon may benefit from RoBERTa's lower false-positive rate only at the cost of recall on rare codes, and RoBERTa may benefit from the lexicon on E2 and unseen wording without adversely impacting precision. This fits Raza et al. (2024), who report that token-level neural tagging is a workable approach to bias detection. It does not show that neural tagging beats rules on PESO text.

### 8.2 Parameters that matter

Sample: 400 ads, seed 20261026, strata = Step 5b route, proportional allocation, minimum 2 per stratum. Pool: 1,100 ads (990 train, 110 validation, split by ad, seed 42). RoBERTa: `roberta-base`, 4 epochs, batch 32, learning rate 3e-5, weight decay 0.01, warm-up 10%, max length 96, seeds 1-3, best epoch by validation exact micro-F1. Intervals: 1,000 bootstrap resamples of ads, seed 7. Full list in `parameters.json`.

## 9. Limitations and their likely effect on the results

| Limitation | Likely effect on the results |
|---|---|
| **Silver labels are rule-derived (circularity).** The lexicon and the labels share the same vocabulary and the same team. | Agreement is probably inflated for the lexicon and for RoBERTa (trained on the labels). Recall 1.000 on E1 and E2 should not be read as validity. Following Kapoor and Narayanan (2023), the scores are not evidence that the system finds real bias. |
| **E3 has only 5 spans, all in the test set.** | RoBERTa recall on E3 is 0 by construction, and the lexicon's 0.2 rests on 5 spans. Neither figure is reliable. The span-level recall of both systems is pulled down by these 5 spans. |
| **Small, unbalanced test positives.** 91 ads and 99 spans, mostly E1 (86). | Intervals are wide (RoBERTa span recall 0.795-0.983). E2 recall (8 spans) and unseen-wording recall (7 spans) are anecdotes. Small differences between systems cannot be resolved. |
| **Employer overlap.** 362 of 400 test ads share an employer with the training pool. | Template reuse can inflate RoBERTa. On the 38 ads with an unseen employer, span F1 falls to 0.789 (lexicon) and 0.695 (RoBERTa), but this subset has 14 positive ads and is too small to confirm a trend. |
| **Silver standard disagrees with the lexicon on "-man" titles, U spans and pronoun pairs.** | Reported false positives (13) and false negatives (4 pair spans) are partly label gaps. The taxonomy v0.3 decisions are still open, so the reference may change. |
| **Single-unit input.** The tagger sees one unit at a time, not the full ad. | Cross-unit cues (for example a gender in the title with a restriction elsewhere) are not used. Both systems may under-detect these cases. |
| **No human gold.** | No claim about validity or agreement with annotators can be made. This is deliberate for this stage and is left to separate work. |
| **One corpus, English-oriented model.** 47 units carry a Filipino marker, and RoBERTa is English-pretrained. | Results may not transfer to other corpora or to code-switched text. |
| **Literature gaps.** We found no verified 2022-2025 source for stratified sampling of job ads, for FPR as a bias-detection metric, or for bootstrap intervals. The RoBERTa paper (2019) is outside the window and was not verified in this session. | These choices rest on standard practice and our own reasoning, not on cited evidence. |

Further and in-depth studies, with human-verified labels and a second corpus, would be needed to confirm the lexicon's validity, especially regarding E2 and E3 and regarding wording not seen in the pool.

## 8. Limitations and their likely effect on the results

(See the interpretation section for the specific effect sizes. The list below fixes the scope of every claim.)

In [14]:
checks = []
def rec(name, ok, detail=""):
    checks.append({"check": name, "passed": bool(ok), "detail": detail})

rec("test sample has 400 distinct ads", len(set(test_ads)) == 400 == len(test_ads))
rec("strata allocation sums to 400", int(alloc.sum()) == 400, str(alloc.to_dict()))
rec("every stratum present in the sample", all(alloc[k] >= 2 for k in alloc.index))
rec("no test ad in training or validation", not (set(train_ads) | val_ads) & set(test_ads))
rec("train + val + test = all 1,500 ads", len(set(train_ads) | val_ads | set(test_ads)) == 1500 == units["record_id"].nunique())
rec("lexicon vocabulary scan used pool units only", set(pool_units["record_id"]).isdisjoint(test_ads))
rec("every gold span reproduces inside its unit text", all(0 <= a < b <= len(txt_of[u]) for u, g in gold_by_unit.items() for a, b, _ in g))
rec("predictions cover every test unit (3 seeds)", all(set(roberta_pred[s]) == set(test_units["unit_id"]) for s in PARAMS["seeds"]))
rec("unit-level counts sum to test units", all(int(A.sum(0)[1].sum()) == len(test_units) for v in SYS.values() for A in v))
rec("ad-level counts sum to 400", all(int(A.sum(0)[0].sum()) == 400 for v in SYS.values() for A in v))
from sklearn.metrics import precision_recall_fscore_support
ad_true = np.array([any(u in gold_by_unit for u in test_units[test_units.record_id == a]["unit_id"]) for a in test_ads])
ad_hat = np.array([any(len(lex_pred[u]) for u in test_units[test_units.record_id == a]["unit_id"]) for a in test_ads])
sk = precision_recall_fscore_support(ad_true, ad_hat, average="binary")[:3]
mine = metric_tab(SYS["Curated lexicon"])[0][0][:3]
rec("ad-level lexicon P/R/F1 match scikit-learn", np.allclose(sk, mine), f"sklearn {np.round(sk, 4)} vs notebook {np.round(mine, 4)}")
rec("results table has every system and level", len(res) == 3 * (3 * 4 + 2 * 3), f"{len(res)} rows")
checks = pd.DataFrame(checks)
checks.to_csv(OUT / "validation_checks.csv", index=False, encoding="utf-8-sig")
display(checks)
print(f"{int(checks['passed'].sum())} of {len(checks)} checks passed.")

summary = {"test_ads": 400, "pool_ads": len(pool_ads), "gold_spans_test": int(tu_gold.sum()),
           "results": {k: {lv: {mn: float(res[(res.system == k) & (res.level == lv) & (res.metric == mn)].value.iloc[0]) for mn in ["precision", "recall", "f1"] + ([] if lv.startswith("span") else ["fpr"])} for lv in LEVELS} for k in SYS}}
json.dump(summary, open(OUT / "results_summary.json", "w"), indent=1)

,check,passed,detail
0,test sample has 400 distinct ads,True,
1,strata allocation sums to 400,True,"{'no span': 196, 'implicit': 92, 'explicit': 91, 'none': 19, 'unclear': 2}"
2,every stratum present in the sample,True,
3,no test ad in training or validation,True,
4,"train + val + test = all 1,500 ads",True,
5,lexicon vocabulary scan used pool units only,True,
6,every gold span reproduces inside its unit text,True,
7,predictions cover every test unit (3 seeds),True,
8,unit-level counts sum to test units,True,
9,ad-level counts sum to 400,True,


12 of 12 checks passed.


## 9. References (2022-2025 unless marked)

Citation status follows `literature-support-for-methods.md`; only what was verified there is cited, and no numeric result of any cited paper is used.

- [1] M. Dikshit, H. Bouamor, and N. Habash, "Investigating gender bias in STEM job advertisements," in *Proc. 5th Workshop Gender Bias Natural Lang. Process. (GeBNLP)*, 2024, pp. 179-189, doi: 10.18653/v1/2024.gebnlp-1.11.
- [4] S. Hu *et al.*, "Balancing gender bias in job advertisements with text-level bias mitigation," *Front. Big Data*, vol. 5, Art. no. 805713, 2022, doi: 10.3389/fdata.2022.805713.
- [5] H. Mihaljevic, I. Muller, K. Dill, and A. Yollu-Tok, "Towards gender-inclusive job postings: A data-driven comparison of augmented writing technologies," *PLOS ONE*, vol. 17, no. 9, Art. no. e0274312, Sep. 2022, doi: 10.1371/journal.pone.0274312.
- [8] T. Jiang, Y. Li, S. Fu, and Y. Chen, "Creating a Chinese gender lexicon for detecting gendered wording in job advertisements," *Inf. Process. Manage.*, vol. 60, no. 5, Art. no. 103424, Sep. 2023, doi: 10.1016/j.ipm.2023.103424.
- [14] S. Kapoor and A. Narayanan, "Leakage and the reproducibility crisis in machine-learning-based science," *Patterns*, vol. 4, no. 9, Art. no. 100804, Sep. 2023, doi: 10.1016/j.patter.2023.100804.
- [23] T. Everitt, P. Ryan, B. Davis, and K. J. Adebayo, "Large language models for detecting bias in job descriptions," *Italian J. Comput. Linguistics*, vol. 10, no. 2, 2024, doi: 10.17454/ijcol102.05. (Introduction and related work read in full text; results not used.)
- [25] R. Frissen, K. J. Adebayo, and R. Nanda, "A machine learning approach to recognize bias and discrimination in job advertisements," *AI & Society*, vol. 38, pp. 1025-1038, 2023, doi: 10.1007/s00146-022-01574-0.
- [26] S. Raza, M. Garg, D. J. Reji, S. R. Bashir, and C. Ding, "Nbias: A natural language processing framework for BIAS identification in text," *Expert Syst. Appl.*, 2024, ScienceDirect pii S0957417423020444. (Article number and DOI to be completed from the publisher page.)
- **Outside the 2022-2025 window, cited only to identify the model:** Y. Liu *et al.*, "RoBERTa: A robustly optimized BERT pretraining approach," arXiv:1907.11692, 2019. Metadata was not re-verified in this session because the arXiv lookup failed.